# 2D Kalman Filter

The purpose of this notebook is for me to map out the concept of Kalman filtering using a concrete example, interpretable time series data and code without using an explicit filtering library.


In [1]:
%pip install numpy

  Using cached numpy-2.5.4-cp314-cp314-macosx_14_0_arm64.whl.metadata (6.6 kB)
Using cached numpy-2.5.4-cp314-cp314-macosx_14_0_arm64.whl (5.5 MB)

[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import csv
import io

In [ ]:
class PlanarKalmanFilter:
    def __init__(self, dt: float):
        self.dt = dt
        self.x = np.zeros((4, 1)) # State vector [x, y, vx, vy]
        self.P = np.eye(4) * 100 # State covariance matrix (initial uncertainty)
        self.F = np.array([[1, 0, dt, 0],  # State transition matrix
                           [0, 1, 0, dt],
                           [0, 0, 1, 0],
                           [0, 0, 0, 1]])
        self.Q = np.eye(4) * 0.1 # Process noise covariance matrix
        self.R = np.eye(4) * 4 # Measurement noise covariance matrix
        self.H = np.array([ # Measurement matrix measuring position components
            [1, 0, 0, 0],
            [0, 1, 0, 0]
        ])

    def predict(self):
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q

        return self.x, self.P

    def update(self, zx: float, zy: float):
        z = np.array([[zx],
                      [zy]])
        y = z - (self.H @ self.x)
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.x = self.x + K @ y
        self.P = (np.eye(4) - K @ self.H) @ self.P

        return self.x, self.P